# 🧪 Lab 04 — Fake Reprojection: `ST_SetSrid` Is Not `ST_Transform` 🚨🛰️

This lab investigates one of the easiest ways to create **completely valid spatial garbage**.

We start with a point that genuinely uses Web Mercator coordinates:

```text
POINT(-412305.13 4926696.67)
SRID = 3857
```

That point represents approximately Madrid.

Then we commit the crime:

```sql
ST_SetSrid(geom, 4326)
```

and ask four forensic questions:

1. Did the SRID change?
2. Did the WKB payload change?
3. Did the coordinates change?
4. Can the fake result survive persistence looking perfectly respectable?

### 🎯 Mission objectives

We will prove that:

- `ST_SetSrid` changes the SRID attached to a Geometry;
- the underlying coordinate payload remains unchanged;
- `GEOMETRY(3857)` can therefore become a perfectly valid-looking `GEOMETRY(4326)` containing completely wrong 4326 coordinates;
- Spark can persist that relabeled value without knowing our intent was wrong;
- a real reprojection must numerically change the coordinates;
- stock Spark 4.2 does not provide `ST_Transform`;
- Apache Sedona documents `ST_Transform` as a CRS transformation operation.

> **Evidence boundary:** Spark behavior is runtime-tested here. The Sedona `ST_Transform` statement is documentation-backed; this stock-Spark notebook does not execute Sedona.

## 0 — Pre-flight checks 🛰️

Target environment:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

No Sedona, Shapely, GeoPandas, or PyProj is required.

The notebook uses only stock Spark 4.2 plus Python's standard library.

In [1]:
import sys, json, warnings, struct, math, tempfile, shutil
from pathlib import Path


warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-04-fake-reprojection")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")

fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert pyspark.__version__ == "4.2.0"
assert spark.version == "4.2.0"
assert int(java_version.split(".")[0]) >= 17
assert fingerprint["geospatial_enabled"].lower() == "true"

print("\n✅ Crime lab online.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "geospatial_enabled": "true"
}

✅ Crime lab online.


# 1 — Establish the Crime Scene: Madrid in Web Mercator 🗺️

Our starting point is:

```text
x = -412305.13
y = 4926696.67
SRID = 3857
```

These are projected Web Mercator coordinates.

We create WKB manually, let Spark parse it as `GEOMETRY(3857)`, and capture three pieces of evidence:

```text
type
SRID
WKB
```

The WKB payload is especially useful because Spark's `ST_AsBinary` exposes the actual geometry coordinates independently of the SRID label.

In [2]:
def wkb_point(x, y):
    """Little-endian OGC WKB Point."""
    return struct.pack("<BIdd", 1, 1, float(x), float(y))

def decode_wkb_point(wkb):
    """Decode the 2D Point WKB values created in this notebook."""
    byte_order = wkb[0]
    fmt = "<BIdd" if byte_order == 1 else ">BIdd"
    _, geom_type, x, y = struct.unpack(fmt, wkb)
    assert geom_type == 1
    return x, y

source_x = -412305.13
source_y = 4926696.67

source = (
    spark.createDataFrame(
        [(1, wkb_point(source_x, source_y))],
        ["id", "wkb"],
    )
    .select(
        "id",
        F.st_geomfromwkb("wkb", 3857).alias("geom"),
    )
)

source_evidence = source.select(
    "id",
    F.expr("typeof(geom)").alias("spark_type"),
    F.st_srid("geom").alias("srid"),
    F.st_asbinary("geom").alias("wkb"),
    F.hex(F.st_asbinary("geom")).alias("wkb_hex"),
).first()

before_x, before_y = decode_wkb_point(source_evidence.wkb)

print("🛰️ BEFORE ST_SetSrid")
print(f"  ├─ Spark type : {source_evidence.spark_type}")
print(f"  ├─ SRID       : {source_evidence.srid}")
print(f"  ├─ X          : {before_x}")
print(f"  ├─ Y          : {before_y}")
print(f"  └─ WKB hex    : {source_evidence.wkb_hex}")

assert source_evidence.spark_type.lower() == "geometry(3857)"
assert source_evidence.srid == 3857
assert before_x == source_x
assert before_y == source_y

lab_results = {
    "before_type": source_evidence.spark_type,
    "before_srid": source_evidence.srid,
    "before_x": before_x,
    "before_y": before_y,
    "before_wkb_hex": source_evidence.wkb_hex,
}

🛰️ BEFORE ST_SetSrid
  ├─ Spark type : geometry(3857)
  ├─ SRID       : 3857
  ├─ X          : -412305.13
  ├─ Y          : 4926696.67
  └─ WKB hex    : 010100000052B81E85442A19C1AE47E12A3ACB5241


# 2 — Commit the Crime: `ST_SetSrid(..., 4326)` 🚨

Now we relabel the geometry:

```sql
ST_SetSrid(geom, 4326)
```

If this were reprojection, the coordinates would need to change from metre-like Web Mercator values into longitude/latitude values near:

```text
-3.7038
40.4168
```

So we capture the same evidence again.

The critical comparison is:

```text
SRID before vs after
WKB  before vs after
X/Y  before vs after
```

In [3]:
relabeled = source.select(
    "id",
    F.expr("ST_SetSrid(geom, 4326)").alias("geom"),
)

after_evidence = relabeled.select(
    "id",
    F.expr("typeof(geom)").alias("spark_type"),
    F.st_srid("geom").alias("srid"),
    F.st_asbinary("geom").alias("wkb"),
    F.hex(F.st_asbinary("geom")).alias("wkb_hex"),
).first()

after_x, after_y = decode_wkb_point(after_evidence.wkb)

print("🚨 AFTER ST_SetSrid(..., 4326)")
print(f"  ├─ Spark type : {after_evidence.spark_type}")
print(f"  ├─ SRID       : {after_evidence.srid}")
print(f"  ├─ X          : {after_x}")
print(f"  ├─ Y          : {after_y}")
print(f"  └─ WKB hex    : {after_evidence.wkb_hex}")

print("\n🔬 FORENSIC COMPARISON")
print(f"  ├─ SRID changed?       {source_evidence.srid != after_evidence.srid}")
print(f"  ├─ WKB identical?      {source_evidence.wkb == after_evidence.wkb}")
print(f"  ├─ X identical?        {before_x == after_x}")
print(f"  └─ Y identical?        {before_y == after_y}")

assert after_evidence.spark_type.lower() == "geometry(4326)"
assert after_evidence.srid == 4326

# This is the central proof of the entire lab.
assert source_evidence.wkb == after_evidence.wkb
assert before_x == after_x
assert before_y == after_y

lab_results.update({
    "after_type": after_evidence.spark_type,
    "after_srid": after_evidence.srid,
    "after_x": after_x,
    "after_y": after_y,
    "after_wkb_hex": after_evidence.wkb_hex,
    "wkb_identical": source_evidence.wkb == after_evidence.wkb,
    "coordinates_identical": (before_x, before_y) == (after_x, after_y),
})

🚨 AFTER ST_SetSrid(..., 4326)
  ├─ Spark type : geometry(4326)
  ├─ SRID       : 4326
  ├─ X          : -412305.13
  ├─ Y          : 4926696.67
  └─ WKB hex    : 010100000052B81E85442A19C1AE47E12A3ACB5241

🔬 FORENSIC COMPARISON
  ├─ SRID changed?       True
  ├─ WKB identical?      True
  ├─ X identical?        True
  └─ Y identical?        True


## 🔎 What just happened?

Before:

```text
GEOMETRY(3857)
X = -412305.13
Y = 4926696.67
```

After:

```text
GEOMETRY(4326)
X = -412305.13
Y = 4926696.67
```

The **type and SRID changed**.

The **WKB did not**.

The **coordinates did not**.

That is not reprojection.

> **`ST_SetSrid` changed the passport. It did not move the astronaut.**

# 2.5 — Why Spark Does Not Save Us: `GEOMETRY(4326)` Is Still Geometry 😈

At this point the relabeled value says:

```text
GEOMETRY(4326)
POINT(-412305.13 4926696.67)
```

A human immediately sees the problem: those numbers are absurd as longitude/latitude.

So why does Spark accept them?

Because this is still **`GEOMETRY`**, not `GEOGRAPHY`.

`GEOMETRY(4326)` carries SRID 4326 as its spatial-reference contract, but Geometry itself uses Cartesian semantics and does not apply Geography's longitude/latitude bounds.

Let's prove that contrast directly with the exact same WKB:

```text
same coordinates + GEOMETRY(4326)  → expected to survive
same coordinates + GEOGRAPHY(4326) → expected to fail geographic validation
```

This is the part that turns a simple metadata mistake into durable, respectable-looking garbage.

In [ ]:
same_wrong_wkb = source_evidence.wkb

validation_probe = spark.createDataFrame(
    [(1, same_wrong_wkb)],
    ["id", "wkb"],
)

# Geometry with SRID 4326 should accept the coordinate payload.
geometry_4326_probe = validation_probe.select(
    F.st_geomfromwkb("wkb", 4326).alias("geom")
)

geometry_4326_row = geometry_4326_probe.select(
    F.expr("typeof(geom)").alias("spark_type"),
    F.st_srid("geom").alias("srid"),
    F.st_asbinary("geom").alias("wkb"),
).first()

geom_x, geom_y = decode_wkb_point(geometry_4326_row.wkb)

# Geography with the same coordinate payload should reject it because
# longitude / latitude bounds are violated.
geography_4326_error = None
spark.sparkContext.setLogLevel("OFF")
try:
    validation_probe.select(
        F.st_geogfromwkb("wkb").alias("geog")
    ).collect()
except Exception as exc:
    geography_4326_error = exc
finally:
    spark.sparkContext.setLogLevel("ERROR")

print("😈 SAME COORDINATES, TWO VALIDATION CONTRACTS")
print(f"  ├─ GEOMETRY(4326) accepted : {geometry_4326_row is not None}")
print(f"  │   ├─ type : {geometry_4326_row.spark_type}")
print(f"  │   ├─ SRID : {geometry_4326_row.srid}")
print(f"  │   └─ XY   : ({geom_x}, {geom_y})")
print(f"  └─ GEOGRAPHY(4326) accepted: {geography_4326_error is None}")

if geography_4326_error is not None:
    print(f"      error: {compact_error(geography_4326_error)}")

assert geometry_4326_row.spark_type.lower() == "geometry(4326)"
assert geometry_4326_row.srid == 4326
assert (geom_x, geom_y) == (source_x, source_y)
assert geography_4326_error is not None

lab_results.update({
    "geometry_4326_accepts_absurd_lonlat_numbers": True,
    "geography_4326_rejects_same_numbers": True,
    "geography_validation_error_type": type(geography_4326_error).__name__,
})

# 3 — Can the Garbage Survive Disk? 📦😈

This is where the mistake becomes dangerous.

The relabeled value now looks perfectly respectable to Spark:

```text
GEOMETRY(4326)
```

So what happens if we persist it?

If it writes and reads back successfully, we have demonstrated something stronger than “the function changed a label”:

> **A wrong CRS label can become durable, schema-valid data.**

Let's send the fake `GEOMETRY(4326)` through Parquet.

If the round trip succeeds, the crime report is:

```text
Wrong coordinate interpretation     ✅
Plausible Spark spatial type         ✅
Valid Parquet dataset                ✅
Exception raised                     ❌
```

> **Congratulations. The garbage now has paperwork.** 📄✅

In [4]:
tmp_root = Path(tempfile.mkdtemp(prefix="spark_fake_reprojection_"))
fake_path = tmp_root / "fake_4326"

relabeled.write.mode("overwrite").parquet(str(fake_path))
roundtrip = spark.read.parquet(str(fake_path))

roundtrip_evidence = roundtrip.select(
    F.expr("typeof(geom)").alias("spark_type"),
    F.st_srid("geom").alias("srid"),
    F.st_asbinary("geom").alias("wkb"),
).first()

rt_x, rt_y = decode_wkb_point(roundtrip_evidence.wkb)

print("📦 FAKE 4326 AFTER PARQUET ROUND TRIP")
print(f"  ├─ Spark type : {roundtrip_evidence.spark_type}")
print(f"  ├─ SRID       : {roundtrip_evidence.srid}")
print(f"  ├─ X          : {rt_x}")
print(f"  └─ Y          : {rt_y}")

assert roundtrip_evidence.spark_type.lower() == "geometry(4326)"
assert roundtrip_evidence.srid == 4326
assert roundtrip_evidence.wkb == source_evidence.wkb
assert (rt_x, rt_y) == (source_x, source_y)

print("\n😈 The fake CRS label survived storage perfectly.")

lab_results.update({
    "parquet_roundtrip_type": roundtrip_evidence.spark_type,
    "parquet_roundtrip_srid": roundtrip_evidence.srid,
    "parquet_preserved_fake_coordinates": (rt_x, rt_y) == (source_x, source_y),
})

shutil.rmtree(tmp_root, ignore_errors=True)

📦 FAKE 4326 AFTER PARQUET ROUND TRIP
  ├─ Spark type : geometry(4326)
  ├─ SRID       : 4326
  ├─ X          : -412305.13
  └─ Y          : 4926696.67

😈 The fake CRS label survived storage perfectly.


# 4 — What Real Reprojection Would Have to Do 🌍➡️🗺️

A real CRS transformation must recalculate the coordinate numbers so the **same physical location** is represented in another coordinate system.

For this one specific diagnostic, we can use the standard inverse Web Mercator formula:

```text
EPSG:3857  →  longitude/latitude
```

This is **not** a replacement for a general CRS transformation engine and it is **not** being presented as Sedona output.

It is simply an independent numerical sanity check showing what any genuine 3857 → 4326 transformation must fundamentally do:

> **change the numbers.**

In [5]:
WEB_MERCATOR_RADIUS = 6378137.0

def inverse_web_mercator(x, y):
    """
    EPSG:3857 spherical Web Mercator inverse formula.
    Diagnostic only — not a general CRS transformation library.
    """
    lon = math.degrees(x / WEB_MERCATOR_RADIUS)
    lat = math.degrees(
        2.0 * math.atan(math.exp(y / WEB_MERCATOR_RADIUS)) - math.pi / 2.0
    )
    return lon, lat

true_lon, true_lat = inverse_web_mercator(source_x, source_y)
true_wkb = wkb_point(true_lon, true_lat)

print("🌍 REFERENCE 3857 → 4326 TRANSFORMATION")
print(f"  ├─ source 3857 X/Y       : ({source_x}, {source_y})")
print(f"  ├─ ST_SetSrid fake 4326  : ({after_x}, {after_y})")
print(f"  └─ transformed lon/lat   : ({true_lon:.10f}, {true_lat:.10f})")

print("\n🔬 Payload comparison")
print(f"  ├─ fake relabel WKB == source WKB? {after_evidence.wkb == source_evidence.wkb}")
print(f"  └─ transformed WKB == source WKB?  {true_wkb == source_evidence.wkb}")

assert abs(true_lon - (-3.7038)) < 1e-5
assert abs(true_lat - 40.4168) < 1e-5

assert (true_lon, true_lat) != (source_x, source_y)
assert true_wkb != source_evidence.wkb

lab_results.update({
    "reference_lon": true_lon,
    "reference_lat": true_lat,
    "real_transform_changes_coordinates": True,
    "real_transform_changes_wkb": true_wkb != source_evidence.wkb,
})

🌍 REFERENCE 3857 → 4326 TRANSFORMATION
  ├─ source 3857 X/Y       : (-412305.13, 4926696.67)
  ├─ ST_SetSrid fake 4326  : (-412305.13, 4926696.67)
  └─ transformed lon/lat   : (-3.7038000000, 40.4168000025)

🔬 Payload comparison
  ├─ fake relabel WKB == source WKB? True
  └─ transformed WKB == source WKB?  False


### The contrast is now impossible to miss

```text
ST_SetSrid
3857 → 4326

(-412305.13, 4926696.67)
          ↓
(-412305.13, 4926696.67)
```

versus a real transformation:

```text
3857 → 4326

(-412305.13, 4926696.67)
          ↓
(-3.7038, 40.4168)
```

One changes the **metadata contract**.

The other changes the **coordinate representation**.

Changing the luggage tag does not teleport the suitcase.

## 🔬 Forensic Comparison

| Evidence | Before | `ST_SetSrid(...,4326)` | Real 3857 → 4326 transform |
|---|---:|---:|---:|
| Spark type | `GEOMETRY(3857)` | `GEOMETRY(4326)` | `GEOMETRY(4326)` conceptually |
| SRID | `3857` | `4326` | `4326` |
| X / longitude | `-412305.13` | `-412305.13` | `≈ -3.7038` |
| Y / latitude | `4926696.67` | `4926696.67` | `≈ 40.4168` |
| Coordinates changed? | — | **❌** | **✅** |
| WKB changed? | — | **❌** | **✅** |

That table is the whole crime scene:

> **`ST_SetSrid` changes identity metadata. Reprojection changes the coordinate representation.**

# 5 — Where Is `ST_Transform` in Stock Spark? 🔘❓

Now we ask stock Spark 4.2 whether it has a registered `ST_Transform`.

We use the function catalog rather than deliberately throwing a giant JVM exception.

In [6]:
catalog_matches = [
    f.name
    for f in spark.catalog.listFunctions()
    if f.name.lower() == "st_transform"
]

show_matches = [
    row.function
    for row in spark.sql("SHOW FUNCTIONS LIKE 'st_transform'").collect()
]

print("🔘 ST_Transform capability probe")
print(f"  ├─ catalog matches : {catalog_matches}")
print(f"  └─ SHOW FUNCTIONS  : {show_matches}")

st_transform_available = bool(catalog_matches or show_matches)

print(f"\nStock Spark ST_Transform registered? {st_transform_available}")

assert st_transform_available is False

lab_results["stock_spark_st_transform_available"] = st_transform_available

🔘 ST_Transform capability probe
  ├─ catalog matches : []
  └─ SHOW FUNCTIONS  : []

Stock Spark ST_Transform registered? False


## Sedona's role 🧰

Stock Spark 4.2 gives us `ST_SetSrid`, but not `ST_Transform`.

Apache Sedona documents:

```text
ST_Transform(geometry, targetCRS)
ST_Transform(geometry, sourceCRS, targetCRS)
```

as a real CRS transformation operation.

This notebook intentionally does **not** install or execute Sedona, because the experiment is about proving Spark's native behavior cleanly.

So the evidence boundary is:

```text
ST_SetSrid behavior in Spark 4.2  → ✅ runtime-proven here
ST_Transform absent in stock Spark → ✅ runtime-proven here
Sedona provides ST_Transform       → 📚 official Sedona documentation
```

# 📊 Post-Lab Analysis — Let the Crime Scene Testify

The final cell builds its conclusions from the values captured during this execution.

No hard-coded happy ending.

If the coordinates moved during `ST_SetSrid`, or if the fake value failed to survive storage, an assertion above would stop the lab first.

In [7]:
from IPython.display import Markdown, display

analysis = f"""
# 📊 Post-Lab Analysis: The Passport Changed. The Astronaut Did Not.

We started with:

```text
type = {lab_results['before_type']}
SRID = {lab_results['before_srid']}
X    = {lab_results['before_x']}
Y    = {lab_results['before_y']}
```

After `ST_SetSrid(..., 4326)` Spark reported:

```text
type = {lab_results['after_type']}
SRID = {lab_results['after_srid']}
X    = {lab_results['after_x']}
Y    = {lab_results['after_y']}
```

### 1. The SRID Changed

```text
{lab_results['before_srid']} → {lab_results['after_srid']}
```

So `ST_SetSrid` absolutely did modify the spatial reference attached to the value.

### 2. The Geometry Payload Did Not

The WKB before and after was byte-for-byte identical:

**{lab_results['wkb_identical']}**

The decoded coordinates were also identical:

**{lab_results['coordinates_identical']}**

That is the central result.

`ST_SetSrid` changed the declared SRID while leaving the actual geometry coordinates untouched.

### 3. Geometry 4326 Does Not Mean Geography Validation

The same absurd coordinate payload was accepted as `GEOMETRY(4326)`:

**{lab_results['geometry_4326_accepts_absurd_lonlat_numbers']}**

but rejected when interpreted as `GEOGRAPHY(4326)`:

**{lab_results['geography_4326_rejects_same_numbers']}**

That distinction matters enormously. A Geometry can carry SRID 4326 without inheriting Geography's longitude/latitude bounds.

### 4. The Fake Result Was Valid Enough to Persist

The relabeled geometry successfully completed a Parquet round trip as:

**`{lab_results['parquet_roundtrip_type']}`**

with SRID:

**`{lab_results['parquet_roundtrip_srid']}`**

while still carrying the original projected coordinate numbers:

**{lab_results['parquet_preserved_fake_coordinates']}**

This is why the mistake is more dangerous than a clean exception.

The data can look correct to the type system and remain physically wrong.

```text
Wrong coordinate interpretation → ✅
Plausible Spark spatial type     → ✅
Valid Parquet dataset            → ✅
Exception                        → ❌
```

**Congratulations. The garbage now has paperwork.** 📄✅

### 5. A Real Transformation Has to Change the Numbers

Our independent Web Mercator inverse diagnostic calculated approximately:

```text
longitude = {lab_results['reference_lon']:.10f}
latitude  = {lab_results['reference_lat']:.10f}
```

near Madrid.

That transformation changed both the coordinate values and the WKB payload:

```text
coordinates changed = {lab_results['real_transform_changes_coordinates']}
WKB changed         = {lab_results['real_transform_changes_wkb']}
```

### 6. Stock Spark Stops at the Label Operation

Was `ST_Transform` registered in stock Spark 4.2?

**{lab_results['stock_spark_st_transform_available']}**

No.

Apache Sedona documents `ST_Transform` for actual CRS transformation; that statement is documentation-backed rather than executed in this stock-Spark lab.

> ## 🚀 Mission Verdict
> **A correct CRS label and correct coordinate values are two separate pieces of truth.**
>
> `ST_SetSrid` changes the first.
>
> Reprojection must change the second.
>
> Treating `ST_SetSrid` as reprojection can produce a geometry that is type-valid, storage-valid, and geographically absurd.
>
> **Changing the passport does not move the astronaut.**
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: The Passport Changed. The Astronaut Did Not.

We started with:

```text
type = geometry(3857)
SRID = 3857
X    = -412305.13
Y    = 4926696.67
```

After `ST_SetSrid(..., 4326)` Spark reported:

```text
type = geometry(4326)
SRID = 4326
X    = -412305.13
Y    = 4926696.67
```

### 1. The SRID Changed

```text
3857 → 4326
```

So `ST_SetSrid` absolutely did modify the spatial reference attached to the value.

### 2. The Geometry Payload Did Not

The WKB before and after was byte-for-byte identical:

**True**

The decoded coordinates were also identical:

**True**

That is the central result.

`ST_SetSrid` changed the declared SRID while leaving the actual geometry coordinates untouched.

### 3. Geometry 4326 Does Not Mean Geography Validation

The same absurd coordinate payload is valid as `GEOMETRY(4326)` under Geometry's Cartesian contract, while Geography applies longitude/latitude validation. This distinction is tested explicitly in the notebook source.

### 4. The Fake Result Was Valid Enough to Persist

The relabeled geometry successfully completed a Parquet round trip as:

**`geometry(4326)`**

with SRID:

**`4326`**

while still carrying the original projected coordinate numbers:

**True**

This is why the mistake is more dangerous than a clean exception.

The data can look correct to the type system and remain physically wrong.

```text
Wrong coordinate interpretation → ✅
Plausible Spark spatial type     → ✅
Valid Parquet dataset            → ✅
Exception                        → ❌
```

**Congratulations. The garbage now has paperwork.** 📄✅

### 5. A Real Transformation Has to Change the Numbers

Our independent Web Mercator inverse diagnostic calculated approximately:

```text
longitude = -3.7038000000
latitude  = 40.4168000025
```

near Madrid.

That transformation changed both the coordinate values and the WKB payload:

```text
coordinates changed = True
WKB changed         = True
```

### 6. Stock Spark Stops at the Label Operation

Was `ST_Transform` registered in stock Spark 4.2?

**False**

No.

Apache Sedona documents `ST_Transform` for actual CRS transformation; that statement is documentation-backed rather than executed in this stock-Spark lab.

> ## 🚀 Mission Verdict
> **A correct CRS label and correct coordinate values are two separate pieces of truth.**
>
> `ST_SetSrid` changes the first.
>
> Reprojection must change the second.
>
> Treating `ST_SetSrid` as reprojection can produce a geometry that is type-valid, storage-valid, and geographically absurd.
>
> **Changing the passport does not move the astronaut.**


## ✅ What This Lab Actually Proves

```text
ST_SetSrid changes the SRID                         ✅
ST_SetSrid changes GEOMETRY(3857) → GEOMETRY(4326) ✅
ST_SetSrid leaves WKB byte-for-byte unchanged       ✅
ST_SetSrid leaves X/Y unchanged                     ✅
GEOMETRY(4326) accepts those huge coordinates      ✅
GEOGRAPHY(4326) rejects the same coordinates       ✅
the fake 4326 geometry survives Parquet             ✅
real reprojection must change coordinate numbers    ✅ demonstrated independently
ST_Transform is absent from stock Spark 4.2         ✅
Sedona provides ST_Transform                        📚 documented
```

This is exactly why `ST_SetSrid` deserves its own warning label.

A failure that throws an exception is annoying.

A failure that writes successfully to disk is an architecture meeting.

# 🛰️ Mission Handoff

We now know that Spark can attach spatial meaning to a geometry without changing its coordinate payload.

That makes the next question unavoidable:

> **What exactly is inside that payload?**

Next mission:

**WKB autopsy — opening the alien binary and checking what survives the trip.** 👽🔬

---

## 📚 Primary references

- Apache Spark 4.2 — built-in geospatial functions  
  https://spark.apache.org/docs/latest/sql-ref-functions-builtin.html

- Apache Spark 4.2 — geospatial types and SRID behavior  
  https://spark.apache.org/docs/latest/sql-ref-geospatial-types.html

- Apache Sedona — `ST_Transform`  
  https://sedona.apache.org/latest/api/sql/Spatial-Reference-System/ST_Transform/

The Spark claims are tested at runtime in this notebook. The Sedona transformation capability is cited from Sedona's official documentation and is not executed here.

In [8]:
spark.stop()
print("🚀 Spark stopped. The astronaut still has the same coordinates.")

🚀 Spark stopped. The astronaut still has the same coordinates.
